# Interactive 3D spline skeleton for synthetic, toy, and high-dimensional datasets

This view projects the fitted skeletal splines into the first three PCA components. Local one-standard-deviation ellipses are drawn in the tangent space orthogonal to each spline and projected into the same ambient coordinates, creating a skeleton with thick bones. The selector includes the synthetic and sklearn toy datasets as well as the real high-dimensional datasets. Rotate and zoom the Plotly figure to inspect the learned backbone, ribs, and local residual thickness.

In [1]:
from pathlib import Path
import warnings
import sys

import numpy as np
from sklearn.datasets import (
    load_breast_cancer,
    load_diabetes,
    load_digits,
    load_wine,
    make_blobs,
    make_circles,
    make_classification,
    make_gaussian_quantiles,
    make_moons,
)

working_dir = Path.cwd().resolve()
notebooks_dir = working_dir / 'notebooks' if (working_dir / 'notebooks' / '__init__.py').exists() else working_dir
project_root = notebooks_dir.parent
if not (project_root / 'src' / 'skeletalembedding').exists():
    raise RuntimeError('Start Jupyter from the repository root or its notebooks/ directory')
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(notebooks_dir))

from skeletalembedding import SkeletalEmbedding
from skeletalembedding.datasets import generate_synthetic_datasets
from skeletalembedding.visualization.interactive import plot_spline_3d

## Fit the skeletal spline network

The selector below covers all synthetic datasets, all 2D sklearn toy datasets (lifted into 3D with independent Z noise), and the real high-dimensional sklearn datasets. Labels are used only for coloring; graph fitting remains unsupervised.

In [2]:
def make_spiral(n_samples=500, noise=0.045, turns=1.15, random_state=5):
    rng = np.random.default_rng(random_state)
    theta = np.linspace(0.0, 2.0 * np.pi * turns, n_samples)
    radius = np.linspace(0.1, 1.0, n_samples)
    points = np.column_stack([radius * np.cos(theta), radius * np.sin(theta)])
    points += rng.normal(scale=noise, size=points.shape)
    return points[rng.permutation(n_samples)]

def lift_planar_dataset(dataset, z_noise=0.045, random_state=0):
    # Add independent Z noise while keeping the signal in the XY plane.
    points, labels = dataset
    points = np.asarray(points, dtype=float)
    if points.shape[1] != 2:
        raise ValueError('lift_planar_dataset expects a two-dimensional dataset')
    rng = np.random.default_rng(random_state)
    points_3d = np.column_stack([
        points,
        rng.normal(scale=z_noise, size=len(points)),
    ])
    return points_3d, np.asarray(labels)

synthetic_datasets_2d = {
    f'synthetic/{name}': (points, np.zeros(len(points), dtype=int))
    for name, points in generate_synthetic_datasets(
        n=500, noise=0.045, random_state=0, binary_tree_depth=3,
    ).items()
}
toy_datasets_2d = {
    'toy/moons': make_moons(n_samples=500, noise=0.07, random_state=0),
    'toy/circles': make_circles(n_samples=500, factor=0.42, noise=0.045, random_state=1),
    'toy/spiral': (make_spiral(), np.zeros(500, dtype=int)),
    'toy/blobs': make_blobs(
        n_samples=500, centers=[(-1.2, -0.8), (0.0, 1.0), (1.2, -0.4)],
        cluster_std=[0.22, 0.28, 0.20], random_state=2,
    ),
    'toy/classification': make_classification(
        n_samples=500, n_features=2, n_redundant=0, n_informative=2,
        n_clusters_per_class=1, class_sep=1.25, flip_y=0.04, random_state=3,
    ),
    'toy/gaussian-quantiles': make_gaussian_quantiles(
        n_samples=500, n_features=2, n_classes=3, random_state=4,
    ),
}
planar_datasets = {**synthetic_datasets_2d, **toy_datasets_2d}
planar_datasets = {
    name: lift_planar_dataset(dataset, random_state=index)
    for index, (name, dataset) in enumerate(planar_datasets.items())
}
high_dim_datasets = {
    'high-dimensional/digits': (load_digits().data, load_digits().target),
    'high-dimensional/wine': (load_wine().data, load_wine().target),
    'high-dimensional/breast-cancer': (load_breast_cancer().data, load_breast_cancer().target),
    'high-dimensional/diabetes': (load_diabetes().data, load_diabetes().target),
}
dataset_catalog = {**planar_datasets, **high_dim_datasets}
list(dataset_catalog)

['synthetic/line',
 'synthetic/star',
 'synthetic/circle',
 'synthetic/figure-eight',
 'synthetic/binary-tree',
 'synthetic/loop-branch',
 'synthetic/polygon-rays-circles',
 'toy/moons',
 'toy/circles',
 'toy/spiral',
 'toy/blobs',
 'toy/classification',
 'toy/gaussian-quantiles',
 'high-dimensional/digits',
 'high-dimensional/wine',
 'high-dimensional/breast-cancer',
 'high-dimensional/diabetes']

## Rotate the 3D spline skeleton

Point color is the digit target. Hover a point for its skeletal-element assignment, longitudinal coordinate, residual norm, and PCA coordinates. Each spline is rendered with sampled 1σ cross-section ellipses.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

dataset_selector = widgets.Dropdown(
    options=list(dataset_catalog),
    value='high-dimensional/digits',
    description='dataset',
    layout=widgets.Layout(width='520px'),
)
render_button = widgets.Button(
    description='Render selected dataset',
    button_style='primary',
    layout=widgets.Layout(width='260px'),
)
smoothness_slider = widgets.FloatSlider(
    value=0.02,
    min=0.0,
    max=0.15,
    step=0.005,
    description='smoothness',
    continuous_update=False,
    readout_format='.3f',
    layout=widgets.Layout(width='240px'),
)
render_output = widgets.Output()

def fit_selected_dataset(name):
    global X, y, model, result, figure
    X, y = dataset_catalog[name]
    is_binary_tree = name == 'synthetic/binary-tree'
    is_planar = name in planar_datasets
    smoothness = float(smoothness_slider.value)
    model = SkeletalEmbedding(
        initialization='legacy_coarsen' if is_binary_tree else 'skeletal',
        n_centroids=36,
        persistence_threshold=4.0 if is_planar else None,
        spline_smoothing=smoothness,
        max_cycles=0 if is_binary_tree else 4,
        random_state=0,
        # Preserve the XY metric for lifted planar data: its small Z noise
        # is measurement thickness, not a reason to rescale the topology.
        standardize=not is_planar,
        persistence_max_points=500 if is_planar else 60,
        spline_samples_per_node=12,
        topology_neighbors=6,
        use_local_pca=True,
        local_pca_neighbors=20,
        use_tangent_boundary_conditions=True,
    )
    with warnings.catch_warnings():
        warnings.filterwarnings(
            'ignore',
            message='Topological landmark constraints could not all be realized by the routing substrate\\.',
            category=RuntimeWarning,
        )
        result = model.fit_transform(X)
    print(
        f'dataset: {name} | observations: {len(X)} | features: {X.shape[1]}\n'
        f'cycles: {model.realized_cycle_count_} | junctions: {len(model.junctions_)} | '
        f'smoothness: {smoothness:.3f} | '
        f'splines: {len(model.splines_)} | ribs: {len(model.rib_paths_)} | '
        f'median residual: {np.median(result.residual_norm):.4f}'
    )
    # Avoid one Plotly trace per unique value for continuous targets.
    unique_labels = np.unique(y)
    plot_labels = y if 1 < len(unique_labels) <= 24 else None
    figure = plot_spline_3d(
        model,
        result,
        labels=plot_labels,
        title=f'{name}: spline skeleton with 1σ tangent-space sections',
        z_scale=1.0,
        point_size=3.5,
        n_spline_samples=24,
        ellipse_bandwidth=0.08,
    )
    figure.show()

def render_selected_dataset(_=None):
    with render_output:
        clear_output(wait=True)
        fit_selected_dataset(dataset_selector.value)

display(widgets.HBox([dataset_selector, smoothness_slider, render_button]))
display(render_output)
render_button.on_click(render_selected_dataset)
smoothness_slider.observe(render_selected_dataset, names='value')
render_selected_dataset()

Output()